<a href="https://colab.research.google.com/github/kundanmrj5-dev/Flyrrank-ml-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kundanmrj5-dev/Flyrrank-ml-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [21]:
!pip -q install duckdb

import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")
if not token:
    raise ValueError("Colab Secret HF_TOKEN is missing. Add it in the Secrets panel.")

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

# Keep the token in the running session; don't print it.
con.execute("SET VARIABLE hf_token = ?", [token])
con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")
del token

FACT = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

print("Connected. March 2026 fact partition selected.")

Connected. March 2026 fact partition selected.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [23]:
feature_frame = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS march_clicks,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_sum_position)
            / NULLIF(SUM(gsc_impressions), 0) AS march_weighted_position
    FROM read_parquet('{FACT}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

display(feature_frame.head())
print("Rows and columns:", feature_frame.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,march_clicks,march_impressions,march_weighted_position
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,0.0,77.0,4.311688
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,22.0,10849.0,8.049866
2,client_62f4a7e64f5e0096,content_e689bc511192751a,0.0,61.0,5.885246
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,1.0,705.0,5.863830
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,0.0,50.0,14.360000


Rows and columns: (176738, 5)


Features: GSC clicks and impressions for March, plus impression-weighted average position when impressions are greater than zero. These are observed during the measurement month.
Label/proxy: My output is a review-priority ranking based on observed search performance; it does not prove a page will decline or improve.
Context: Report date, anonymized client ID, and anonymized content ID. I use IDs to group records, not as predictive features.
Excluded: Unavailable GSC rows, position values of zero as real measurements, future data, private information, and my Hugging Face token. Missing data is not treated as zero performance.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [25]:
q1 = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS duplicate_rows
    FROM read_parquet('{FACT}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

display(q1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_rows


In [26]:
q2 = con.execute(f"""
    SELECT COUNT(*) AS march_rows,
           MIN(report_date) AS first_date,
           MAX(report_date) AS last_date
    FROM read_parquet('{FACT}')
""").df()

display(q2)

,march_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [27]:
q3 = con.execute(f"""
    SELECT COUNT(*) AS all_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
    FROM read_parquet('{FACT}')
""").df()

display(q3)

,all_rows,available_rows
0,9841378,3611061


In [28]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score

# Define a simple March proxy outcome:
# 1 = at least 100 impressions but zero clicks; 0 = otherwise.
demo = feature_frame.dropna(
    subset=["march_weighted_position"]
).copy()

demo["outcome_proxy"] = (
    (demo["march_impressions"] >= 100) &
    (demo["march_clicks"] == 0)
).astype(int)

if demo["outcome_proxy"].nunique() < 2:
    raise ValueError("The proxy has only one class; check the March data.")

# Keep each client entirely in either train or test.
splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.2, random_state=42
)
train_idx, test_idx = next(
    splitter.split(
        demo,
        demo["outcome_proxy"],
        groups=demo["client_hash_id"]
    )
)

y_train = demo.iloc[train_idx]["outcome_proxy"]
y_test = demo.iloc[test_idx]["outcome_proxy"]

# Baseline comparison: position only.
clean_model = DecisionTreeClassifier(max_depth=3, random_state=42)
clean_model.fit(
    demo.iloc[train_idx][["march_weighted_position"]],
    y_train
)
honest_score = balanced_accuracy_score(
    y_test,
    clean_model.predict(
        demo.iloc[test_idx][["march_weighted_position"]]
    )
)

# Deliberately leak the answer into the inputs.
demo["deliberate_leak"] = demo["outcome_proxy"]

leaky_model = DecisionTreeClassifier(max_depth=3, random_state=42)
leaky_model.fit(
    demo.iloc[train_idx][["march_weighted_position", "deliberate_leak"]],
    y_train
)
leaked_score = balanced_accuracy_score(
    y_test,
    leaky_model.predict(
        demo.iloc[test_idx][["march_weighted_position", "deliberate_leak"]]
    )
)

print(f"Without leaked column (comparison score): {honest_score:.3f}")
print(f"With deliberate leakage (invalid score): {leaked_score:.3f}")

# Remove the leaked column. Do not use the leaked score as model performance.
del demo["deliberate_leak"]

Without leaked column (comparison score): 0.500
With deliberate leakage (invalid score): 1.000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [29]:
available = int(q3.loc[0, "available_rows"])
total = int(q3.loc[0, "all_rows"])

print(f"GSC-available rows: {available:,} of {total:,}")
print(f"Availability: {available / total:.1%}")
print("Unavailable rows are excluded, not treated as zero performance.")

GSC-available rows: 3,611,061 of 9,841,378
Availability: 36.7%
Unavailable rows are excluded, not treated as zero performance.


Client data history and GSC availability differ. This March snapshot cannot prove that refreshing a page causes better performance or predict future results. A position value of zero is not a valid position measurement, and unavailable data is not zero performance. A person should review suggestions before taking action.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.